<a href="https://colab.research.google.com/github/IndiraTejaswini/Pytorch_Neural_networks/blob/main/Decision_Tree_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

data = load_iris()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    prob = counts / len(y)
    return 1 - np.sum(prob ** 2)

In [ ]:
def split(X, y, feature, threshold):
    left = X[:, feature] <= threshold
    right = X[:, feature] > threshold

    return X[left], X[right], y[left], y[right]

In [ ]:
def best_split(X, y):
    best_gini = float("inf")
    best_feature = None
    best_threshold = None

    for feature in range(X.shape[1]):
        thresholds = np.unique(X[:, feature])

        for t in thresholds:
            X_l, X_r, y_l, y_r = split(X, y, feature, t)

            if len(y_l) == 0 or len(y_r) == 0:
                continue

            gini_l = gini(y_l)
            gini_r = gini(y_r)

            weighted = (len(y_l)/len(y))*gini_l + (len(y_r)/len(y))*gini_r

            if weighted < best_gini:
                best_gini = weighted
                best_feature = feature
                best_threshold = t

    return best_feature, best_threshold

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

In [ ]:
class DecisionTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth

    def fit(self, X, y):
        self.root = self._build(X, y, depth=0)

    def _build(self, X, y, depth):
        # Stop conditions
        if len(set(y)) == 1 or depth >= self.max_depth:
            return Node(value=self._most_common(y))

        feature, threshold = best_split(X, y)

        if feature is None:
            return Node(value=self._most_common(y))

        X_l, X_r, y_l, y_r = split(X, y, feature, threshold)

        left = self._build(X_l, y_l, depth + 1)
        right = self._build(X_r, y_r, depth + 1)

        return Node(feature, threshold, left, right)

    def _most_common(self, y):
        return max(set(y), key=list(y).count)

    def predict(self, X):
        return [self._predict(x, self.root) for x in X]

    def _predict(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._predict(x, node.left)
        else:
            return self._predict(x, node.right)

In [ ]:
model = DecisionTree(max_depth=3)
model.fit(X_train, y_train)

In [ ]:
preds = np.array(model.predict(X_test))

accuracy = np.sum(preds == y_test) / len(y_test)
print("Accuracy:", accuracy)

Accuracy: 1.0
